# Описание задачи "Разработка нефтегазовых месторождений"

Набор данных содержит 442 о различных нефтегазовых месторождениях.
Тренировочный набор - 309 строк.
Тестовый набор - 133 строк.

Каждое месторождение обладает 19 параметрами:
1. Field name - название месторождения
2. Reservoir unit - юнит месторождения
3. Country - страна расположения
4. Region - регион расположения
5. Basin name - название бассейна пород
6. Tectonic regime - тектонический режим
7. Latitude - широта
8. Longitude - долгота
9. Operator company - название компании
10. Onshore or oﬀshore - на суше или нет
11. Hydrocarbon type (main) - тип углеводорода
12. Reservoir status (current) - статус месторождения
13. Structural setting - структурные свойства
14. Depth (top reservoir ft TVD) - глубина
15. Reservoir period - литологический период
16. Lithology (main) - литология
17. Thickness (gross average ft) - общая толщина
18. Thickness (net pay average ft) - эффективная толщина
19. Porosity (matrix average.. - пористость
20. Permeability (air average mD) – проницаемость

**Что нужно сделать**:

Принять участие в соревновании на Kaggle:
Оно доступно по [ссылке](https://www.kaggle.com/competitions/classification-of-oil-and-gas/submissions#).

Разработать и оформить решение в ноутбуке:
* Исследование и анализ датасета.
* Предобработка данных.
* Feature Engineering (если необходимо).
* Подбор признаков, их анализ и оценка важности.
* Обучение нескольких моделей, их сравнение.
* Подбор гиперпараметров.
* Выбор лучшей модели и объяснение выбора.
* Предсказание на тестовых данных.


|**Критерии оценивания**||
|:---|:---|
|Анализ данных и их предобработка	|0-3 балла|
|Feature Engineering и отбор признаков	|0-2 балла|
|Выбор и обучение нескольких моделей	|0-3 балла|
|Подбор гиперпараметров и объяснение выбора лучшей модели	|0-2 балла|
|Результат на Kaggle	|0-1 балл|
|Качество оформления ноутбука (чистый код, комментарии, выводы)	|0-1 балл|
|**Итого**	|**12 баллов**|


# Считывание данных

In [1]:
import opendatasets as od

# Загрузим датасет на прямую с kaggle
# Для автомтическй загрузки надо положить файл kaggle.json в папку с ноутбуком
# Файл скачивается в настройках Kaggle в разделе Legacy API Credentials
dataset_url = 'https://www.kaggle.com/competitions/classification-of-oil-and-gas/data'

od.download(dataset_url)

ModuleNotFoundError: No module named 'opendatasets'

In [ ]:
# иморитирование всех необходимых библиотек
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Считываем тренировочные данные
train = pd.read_csv("./classification-of-oil-and-gas/train_oil.csv")
test = pd.read_csv("./classification-of-oil-and-gas/oil_test.csv")

print(f"Train dataset shape: {train.shape}")
print(f"Test dataset shape: {test.shape}")

In [ ]:
# Посмотрим как выглядят данные
train.head(5)

# Иследование и обработка данных (EDA)

## Общая информация

In [ ]:
# процент дублей 
(len(train) - len( train.drop_duplicates() )) * 100 / len(train)

Дублирующих строк нет.

In [ ]:
train.info()

In [ ]:
test.info()

Видно, что тестовая выборка достаточно репрезентативна (133/)

In [ ]:
# Выведем версию Pandas (в 3.0 - выводит тип str, а не object)
print(f"Версия Pandas: {pd.__version__}")

## Обработка пропущенных значений

### Обзор

Есть пропущенные значения в тренировочных и тестовых данных. Визуализируем пропуски.

In [ ]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

In [ ]:
train.isna().sum().sum()

Поле Field name (название месторождения) у нас везде заполнено и если оно будет встречаться и там, и там, то по нему мы попробуем востановить пропущенные значения.

In [ ]:
def fill_na(train_df, check_col, test_df):
    """Принимает train, поле, по которому надо свернуть, чтобы получить "средние", test.
    Возвращает: пересечние, где нет значений и где есть в train, заполненные ими train, test.
    """
    
    # Разделим данные на две таблицы, где есть пропуски и где все поля заполнены
    
    # .any(axis='columns') - перебирает все строки и помечает всю строку True, если есть хотя бы один NaN в любой колонке
    missed = train_df[train_df.isna().any(axis='columns')] 
    
    filled = train_df.dropna()
    
    # Получим список заполненных полей    
    filled_names = filled[check_col].unique()
    
    # Отберём те строки, в таблице с пропусками, которые есть без пропусков
    common_names = missed[missed[check_col].isin(filled_names)]
    
    # Выведем результат из общей таблицы, чтобы визауально оценить возможность заполнения пропусков
    visual_check = train_df[train_df[check_col].isin(common_names[check_col])].sort_values(check_col)

    # создадим таблицу справочник с модой и медианой по названию месторождения
    mapping = filled.groupby(check_col).agg({
        'Country': lambda x: x.mode().iat[0], # так как мода вычисляется, то для указания индекса используем iat 
        'Region': lambda x: x.mode().iat[0],
        'Basin name': lambda x: x.mode().iat[0],
        'Latitude': 'median',
        'Longitude':'median',
    })
        
    # Заполнение, сбрасываем индекс на название, заполняем из справочника, восстанавливаем индекс
    train_df_filled = train_df.set_index(check_col).fillna(mapping).reset_index()
    test_df_filled = test_df.set_index(check_col).fillna(mapping).reset_index()

    return visual_check, train_df_filled, test_df_filled
        

### Fill Nan by *Field name*

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Field name', test)

In [ ]:
filled_train.isna().sum().sum()

In [ ]:
visual_check

Видно, что можем смело заполнть пропуски по уже имеющимся данным.

In [ ]:
# Проверим заполнение.

# как было
visual_check.head(2)

In [ ]:
# вывод как заполнилось
filled_train.loc[[218, 230]]

In [ ]:
# Применяем изменения
train, test = filled_train, filled_test

In [ ]:
train.isna().sum().sum()

### Fill Nan by *Reservoir unit*

Теперь попробуем определить по полю *Reservoir unit*

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Reservoir unit', test)

filled_train.isna().sum().sum()

In [ ]:
visual_check

In [ ]:
# Проверим заполнение.

# как было
indices = visual_check.tail(3).index
display(visual_check.tail(3))
# как заполнилось
filled_train.loc[indices]

Есть поле *UNNAMED*, по которому восстановление не получится. Проверим есть ли у этого оператора другие месторождения.

In [ ]:
train[train['Operator company'] == 'ORENBURGGAZPROM']

Заменим это поле на уникальное

In [ ]:
mask = train['Reservoir unit'] == 'UNNAMED'
train.loc[mask, 'Reservoir unit'] = train.loc[mask, 'Field name'] + ' UNNAMED'


Сделаем ещё раз

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Reservoir unit', test)

display(filled_train.isna().sum().sum())

# Проверим заполнение.

# как было
indices = visual_check.head(2).index
display(visual_check.head(2))
# как заполнилось
filled_train.loc[indices]

In [ ]:
# Применяем изменения
train, test = filled_train, filled_test

train.isna().sum().sum()

### Fill Nan by *Basin name*

In [ ]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

In [ ]:
train[train['Basin name'].notna() & train.isna().any(axis='columns')]

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Basin name', test)

display(filled_train.isna().sum().sum())

# Проверим заполнение.

# как было
indices = visual_check.head().index
display(visual_check.head())
# как заполнилось
filled_train.loc[indices]

In [ ]:
# Применяем изменения
train, test = filled_train, filled_test

train.isna().sum().sum()

### Fill Nan by *Country*

Видно, что кое-где есть страна, но некоторые другие поля не заполнены.

In [ ]:
train[train['Country'].notna() & train.isna().any(axis='columns')]

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Country', test)

filled_train.isna().sum().sum()

In [ ]:
indices = train[train['Country'].notna() & train.isna().any(axis='columns')].index

In [ ]:
filled_train.loc[indices]

In [ ]:
visual_check

In [ ]:
train, test = filled_train, filled_test

### Fill Nan by *Operator company*

In [ ]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

In [ ]:
indices_all = train[train.isna().any(axis='columns')].index
train[train.isna().any(axis='columns')]

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Operator company', test)

display(filled_train.isna().sum().sum())

# Проверим заполнение.

# как было
indices = visual_check.tail().index
display(visual_check.tail())
# как заполнилось
filled_train.loc[indices]

In [ ]:
train, test = filled_train, filled_test

### Fill Nan by *Structural setting*

In [ ]:
# Выведем список колонок с пропущенными значениями
train.isna().sum()[train.isna().sum() > 0]

In [ ]:
visual_check, filled_train, filled_test = fill_na(train, 'Structural setting', test)

display(filled_train.isna().sum().sum())

# Проверим заполнение.

# как было
indices = visual_check.tail().index
display(visual_check.tail())
# как заполнилось
filled_train.loc[indices]

In [ ]:
train, test = filled_train, filled_test

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
test[test.isna().any(axis=1)]

## Обработка категориальных признаков

### Обзор

In [ ]:
train.describe(include='all').T

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train['Onshore/Offshore'] = le.fit_transform(train['Onshore/Offshore'])


# Финальный сабмит

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
display(train.shape)

display(test.shape)

In [ ]:
train = train.dropna()

display(train.shape)

In [ ]:
# test['Latitude'] = test['Latitude'].fillna(0)
# test['Longitude'] = test['Longitude'].fillna(0)
test = test.fillna(0)

In [ ]:
train.info()

In [ ]:
# Получение списка имен категориальных колонок
categorical_cols = train.drop('Onshore/Offshore', axis='columns').select_dtypes(include=['object', 'string']).columns.tolist()

categorical_cols


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Список категориальных столбцов
categ = categorical_cols

# Создаем OrdinalEncoder, обрабатываем неизвестные категории специальным значением -1
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

train_encoded = encoder.fit_transform(train[categ]) # numpy
test_encoded = encoder.transform(test[categ]) # numpy

# Заменяем старые категориальные столбцы новыми закодированными
train[categ] = train_encoded
test[categ] = test_encoded

In [ ]:
display(train.shape)

display(test.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Создадим модель дерева решений
tree = DecisionTreeClassifier(random_state=42)
knn = KNeighborsClassifier(2)

X = train.drop(columns=['Onshore/Offshore'])
y = train['Onshore/Offshore']
y = le.fit_transform(y)

# обучение модели
tree.fit(X, y)

# предсказание ответов для тестовой выборки
y_pred_tree = tree.predict(test)

y_pred_tree

In [ ]:
y_pred_tree.sum()

In [ ]:
ans_df = pd.DataFrame(y_pred_tree, columns=['Onshore/Offshore'])

ans_df.reset_index(inplace=True)
ans_df.to_csv('./classification-of-oil-and-gas/submition_baseline-fillna_OrdinalEncoder.csv', index=False)

In [ ]:
train.describe()

In [ ]:
pd.__version__